# 03 — Typed tools and tool choice
A tool is a bounded Python capability the model can request. First test the
plain function, then watch the CrewAI agent decide when to use its wrapper.



## Problem

The model cannot calculate from a trusted function unless that function is exposed.


## Concept

A CrewAI tool wraps a typed Python contract the agent may invoke.

CrewAI exposes a Python function through a schema and a natural-language description. The model sees the contract and can request a call; the runtime validates arguments and executes the function.


## Abstraction

task → agent chooses ROI tool → validated inputs → scenario result. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
Crew Task ──▶ Agent ──▶ typed ROI tool request
                  ▲                    │
                  │               Python validation
                  │                    │
                  └──── observation ◀──┘
                 Agent ──▶ scenario answer + caveat
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Test the pure ROI function and its invalid-input contract before involving the model

Test the pure ROI function and its invalid-input contract before involving the model.


In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, Crew, Process, Task
from crewai.tools import tool

from workshop.common import model
from workshop.tools import calculate_annual_roi

print("Pure function:", calculate_annual_roi(45_000, 30_000))
try:
    calculate_annual_roi(45_000, 0)
except ValueError as error:
    print("Input contract:", error)



## Wrap the tested function as a CrewAI tool
The docstring describes when the agent should use this capability. The
function still validates inputs in Python, independent of model behavior.



## Step 2 — Wrap that function with CrewAI's @tool decorator and write a precise description

Wrap that function with CrewAI's @tool decorator and write a precise description.


In [ ]:
@tool("Calculate annual ROI")
def roi_tool(annual_benefit: float, annual_cost: float) -> str:
    """Calculate a hypothetical annual ROI percentage from stated USD assumptions."""
    value = calculate_annual_roi(annual_benefit, annual_cost)
    return json.dumps({"roi_percent": value, "status": "scenario, not a guarantee"})


print("Available tool:", roi_tool.name)



## Give only the required tool to the agent
The calculation is a scenario, not a measured financial outcome.



## Step 3 — Give the agent only the needed tool, then distinguish the numeric tool result from the model's narrative

Give the agent only the needed tool, then distinguish the numeric tool result from the model's narrative.


In [ ]:
analyst = Agent(
    role="ROI Analyst",
    goal="Calculate transparent planning scenarios",
    backstory="You label estimates and assumptions.",
    llm=model(),
    tools=[roi_tool],
    max_iter=3,
    allow_delegation=False,
    verbose=True,
)
task = Task(
    description="Use the ROI tool for $45,000 annual benefit and $30,000 "
    "annual cost. Explain why the result is only a planning scenario.",
    expected_output="ROI percentage and assumption caveat.",
    agent=analyst,
)
print(Crew(agents=[analyst], tasks=[task], process=Process.sequential).kickoff().raw)


## Execution flow

task → agent chooses ROI tool → validated inputs → scenario result. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

The model selects tools from their names/descriptions, but Python must still reject invalid arguments. Test the pure function before involving the model.


## Production implication

Expose only needed tools, validate arguments inside Python, and limit permissions. Tool descriptions are part of the interface contract.


## Your turn

Try a zero annual cost in the pure function before the agent run.

**Checkpoint:** Invalid cost fails, and the model labels the valid ROI as a scenario.
